# `evaluate.py` — Batch Evaluation Script

## Purpose
Runs **batch evaluation** of the trained model on the full test (or val) split and writes structured results to the `evaluation/` directory.

## Difference from `inference.py`
| | `evaluate.py` | `inference.py` |
|-|-|-|
| Input | Full CSV split (batch) | Single review text |
| Output | JSON metrics + confusion matrices + CSV predictions | Single prediction dict |
| Use-case | Reproducing paper results, comparing checkpoints | Website API, interactive XAI |

## Usage
```bash
# From ml-research root:
python inference_bridge/evaluate.py \
    --checkpoint outputs/cosmetic_sentiment_v1/best_model.pt \
    --split test \
    --save_dir inference_bridge
```

## Output Files
| File | Description |
|------|-------------|
| `metrics.json` | All per-aspect and overall Macro-F1, Accuracy, MCC etc. |
| `predictions.csv` | Full prediction table (text, aspect, true_label, pred_label) |
| `all_confusion_matrices.png` | Grid of 7 confusion matrices (one per aspect) |
| `error_analysis.csv` | Misclassified samples for manual inspection |
| `mixed_sentiment_analysis.json` | MSR-specific metrics (mixed_detection_rate etc.) |

In [1]:
print("⏳ Starting: Running MSR evaluation...")
import time
_start_time = time.time()
import os, yaml, torch, argparse, sys
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

ML_RESEARCH = os.path.dirname(os.path.abspath(''))
os.chdir(ML_RESEARCH)  # ml-research/
SRC_DIR = os.path.join(ML_RESEARCH, 'src')
EVAL_DIR = os.path.join(ML_RESEARCH, 'outputs', 'cosmetic_sentiment_v1', 'evaluation')
for _p in tqdm([SRC_DIR, EVAL_DIR], desc="Processing _p"):
    if _p not in sys.path: sys.path.insert(0, _p)

from models.model import create_model
from utils.data_utils  import (DependencyParser, DependencyParsingDataset,
                                CosmeticReviewDataset, collate_fn_with_dependencies)
from utils.metrics     import AspectSentimentEvaluator, MixedSentimentEvaluator
from transformers      import RobertaTokenizer
from torch.utils.data  import DataLoader
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Running MSR evaluation.")


⏳ Starting: Running MSR evaluation...


Processing _p: 100%|██████████| 2/2 [00:00<?, ?it/s]


🕒 Done in 6.54s
✅ Completed: Running MSR evaluation.


## Loading the Checkpoint

In [2]:
print("⏳ Starting: Defining function load_model...")
import time
_start_time = time.time()
def load_model(checkpoint_path, device='cuda'):
    """
    Load a trained model from a checkpoint file.
    
    The checkpoint is a dict saved by ExperimentTrainer containing:
      - 'model_state_dict': the trained weights
      - 'config': the full training+model config (so model can be recreated without a separate YAML)
      - 'best_val_metric': the best val Macro-F1 achieved during training
    """
    print("📥 Loading a saved model checkpoint...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config     = checkpoint['config']

    print("🧩 Building the neural network architecture...")
    model = create_model(config)                              # Rebuild model architecture
    model.load_state_dict(checkpoint['model_state_dict'])    # Load trained weights
    model.to(device).eval()                                  # GPU + eval mode (disable dropout)

    best_metric = checkpoint.get('best_val_metric', 0.0)     # Use .get() to handle old checkpoints
    print(f'Model loaded from {checkpoint_path} — best_val_metric: {best_metric:.4f}')
    return model, config
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function load_model.")


⏳ Starting: Defining function load_model...
🕒 Done in 0.00s
✅ Completed: Defining function load_model.


## Batch Prediction Loop

In [ ]:
print("⏳ Starting: Defining standalone visualization tools...")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_curve, auc
import pandas as pd
from collections import Counter


def print_results(results_dict, aspect_name):
    """Full formatted result table for one aspect."""
    if aspect_name not in results_dict:
        print(f"[Evaluator] No results for '{aspect_name}'")
        return

    results = results_dict[aspect_name]
    print(f"\n{'='*70}")
    print(f"Results for {aspect_name.upper()}")
    print(f"{'='*70}")
    print(f"Accuracy:      {results['accuracy']:.4f}")
    print(f"Macro F1:      {results['macro_f1']:.4f}")
    if results.get('roc_auc'):
        print(f"ROC AUC:       {results['roc_auc']:.4f}")
    print(f"Weighted F1:   {results['weighted_f1']:.4f}")
    print(f"MCC:           {results['mcc']:.4f}")

    print(f"\n{'Class':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Support'}")
    print(f"{'-'*70}")
    for i, name in enumerate(['Negative', 'Neutral', 'Positive']):
        print(f"{name:<12} {results['per_class_precision'][i]:<12.4f} "
              f"{results['per_class_recall'][i]:<12.4f} "
              f"{results['per_class_f1'][i]:<12.4f} {int(results['support'][i])}")

    print(f"\nConfusion Matrix:")
    print("              Pred Neg  Pred Neu  Pred Pos")
    for row_name, row_idx in [("True Neg", 0), ("True Neu", 1), ("True Pos", 2)]:
        print(f"{row_name}      "
              f"{results['confusion_matrix'][row_idx][0]:8d}  "
              f"{results['confusion_matrix'][row_idx][1]:8d}  "
              f"{results['confusion_matrix'][row_idx][2]:8d}")

def plot_confusion_matrix(results_dict, aspect_name, save_path=None):
    if aspect_name not in results_dict:
        print(f"[Evaluator] No results for '{aspect_name}'")
        return
    cm = results_dict[aspect_name]['confusion_matrix']
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Neutral', 'Positive'],
                yticklabels=['Negative', 'Neutral', 'Positive'])
    plt.title(f'Confusion Matrix — {aspect_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"[Evaluator] Confusion matrix saved to {save_path}")
    else:
        plt.show()
    plt.close()

def plot_all_confusion_matrices(results_dict, save_dir=None):
    if not results_dict:
        print("[Evaluator] No results to plot")
        return
    n_aspects = len(results_dict)
    n_cols    = 3
    n_rows    = (n_aspects + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    axes = axes.flatten() if n_aspects > 1 else [axes]

    print(f"[Evaluator] Plotting {n_aspects} confusion matrices...")
    for idx, (aspect_name, results) in enumerate(results_dict.items()):
        cm = results['confusion_matrix']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Neg', 'Neu', 'Pos'],
                    yticklabels=['Neg', 'Neu', 'Pos'],
                    ax=axes[idx], cbar=False)
        axes[idx].set_title(f'{aspect_name}\nF1={results["macro_f1"]:.3f}')
        axes[idx].set_ylabel('True')
        axes[idx].set_xlabel('Predicted')

    for idx in range(n_aspects, len(axes)):
        axes[idx].axis('off')
    plt.tight_layout()

    if save_dir:
        save_path = Path(save_dir) / 'all_confusion_matrices.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"[Evaluator] All confusion matrices saved to {save_path}")
    else:
        plt.show()
    plt.close()

def plot_roc_curve(results_dict, aspect_name, save_path=None):
    """Plot multiclass ROC curve (One-vs-Rest) for a single aspect."""
    if aspect_name not in results_dict or results_dict[aspect_name].get('y_prob') is None:
        print(f"[Evaluator] No probability results for '{aspect_name}'")
        return
    
    info = results_dict[aspect_name]
    y_true, y_prob = info['y_true'], info['y_prob']
    class_names = ['Negative', 'Neutral', 'Positive']
    colors = ['red', 'gray', 'green']
    
    plt.figure(figsize=(10, 8))
    for i in range(3):
        # One-vs-Rest: Treat current class as positive, all others as negative
        y_true_binary = (np.array(y_true) == i).astype(int)
        # Only calculate if there's at least one instance of the class
        if np.sum(y_true_binary) > 0:
            fpr, tpr, _ = roc_curve(y_true_binary, y_prob[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, color=colors[i], lw=2, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
        
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    
    macro_auc = info.get('roc_auc', 0)
    plt.title(f'ROC Curve — {aspect_name} (Macro AUC: {macro_auc:.3f})')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"[Evaluator] ROC curve saved to {save_path}")
    else:
        plt.show()
    plt.close()

def plot_all_roc_curves(results_dict, save_dir=None):
    """Plot a grid of ROC curves for all aspects."""
    if not results_dict:
        return
    
    prob_aspects = [asp for asp, res in results_dict.items() if res.get('y_prob') is not None]
    n_aspects = len(prob_aspects)
    if n_aspects == 0:
        print("[Evaluator] No probability results found for ROC analysis")
        return
        
    n_cols = 3
    n_rows = (n_aspects + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
    axes = axes.flatten() if n_aspects > 1 else [axes]
    
    colors = ['red', 'gray', 'green']
    class_names = ['Neg', 'Neu', 'Pos']
    
    plot_idx = 0
    for aspect_name in prob_aspects:
        res = results_dict[aspect_name]
        y_true, y_prob = res['y_true'], res['y_prob']
        ax = axes[plot_idx]
        
        for i in range(3):
            y_true_binary = (np.array(y_true) == i).astype(int)
            if np.sum(y_true_binary) > 0:
                fpr, tpr, _ = roc_curve(y_true_binary, y_prob[:, i])
                roc_auc = auc(fpr, tpr)
                ax.plot(fpr, tpr, color=colors[i], label=f'{class_names[i]}: {roc_auc:.2f}')
            
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
        macro_auc = res.get('roc_auc', 0)
        ax.set_title(f'ROC: {aspect_name} (AUC: {macro_auc:.3f})')
        ax.set_ylim([0, 1.05])
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)
        plot_idx += 1
        
    for i in range(plot_idx, len(axes)):
        axes[i].axis('off')
        
    plt.tight_layout()
    if save_dir:
        save_path = Path(save_dir) / 'all_roc_curves.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"[Evaluator] All ROC curves saved to {save_path}")
    else:
        plt.show()
    plt.close()

def compare_aspects(results_dict):
    if not results_dict:
        print("[Evaluator] No results yet")
        return

    print(f"\n{'='*90}")
    print("Performance Comparison Across Aspects")
    print(f"{'='*90}")
    print(f"{'Aspect':<15} {'Accuracy':<12} {'Macro-F1':<12} {'Weighted-F1':<12} {'MCC':<12}")
    print(f"{'-'*90}")

    for aspect_name in sorted(results_dict.keys()):
        r = results_dict[aspect_name]
        print(f"{aspect_name:<15} {r['accuracy']:<12.4f} "
              f"{r['macro_f1']:<12.4f} {r['weighted_f1']:<12.4f} {r['mcc']:<12.4f}")

    aspects = list(results_dict.keys())
    metrics_to_plot = ['accuracy', 'macro_f1', 'weighted_f1', 'mcc']
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()
    for idx, metric in enumerate(metrics_to_plot):
        values = [results_dict[asp][metric] for asp in aspects]
        axes[idx].bar(aspects, values, color='steelblue', alpha=0.7)
        axes[idx].set_title(metric.replace('_', ' ').title())
        axes[idx].set_ylabel('Score')
        axes[idx].set_ylim([0, 1])
        axes[idx].tick_params(axis='x', rotation=45)
        for i, v in enumerate(values):
            axes[idx].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()

def generate_latex_table(results_dict):
    if not results_dict:
        return ""
    latex  = "\\begin{table}[h]\n\\centering\n"
    latex += "\\begin{tabular}{lcccc}\n\\hline\n"
    latex += "Aspect & Accuracy & Macro F1 & Weighted F1 & MCC \\\\\n\\hline\n"
    for aspect in sorted(results_dict.keys()):
        r = results_dict[aspect]
        latex += (f"{aspect} & {r['accuracy']:.4f} & {r['macro_f1']:.4f} & "
                  f"{r['weighted_f1']:.4f} & {r['mcc']:.4f} \\\\\n")
    latex += "\\hline\n\\end{tabular}\n"
    latex += "\\caption{Multi-aspect sentiment analysis results}\n"
    latex += "\\label{tab:results}\n\\end{table}"
    return latex

class ErrorAnalyzer:
    """Analyze prediction errors for improvement insights"""
    def __init__(self, aspect_names, class_names):
        self.aspect_names = aspect_names
        self.class_names  = class_names

    def analyze_errors(self, texts, y_true, y_pred, aspects, save_path=None):
        print(f"\n[ErrorAnalyzer] Analyzing {len(texts)} predictions...")

        errors = [
            {
                'text'      : texts[i],
                'aspect'    : aspects[i],
                'true_label': self.class_names[y_true[i]],
                'pred_label': self.class_names[y_pred[i]],
                'error_type': f"{self.class_names[y_true[i]]}->{self.class_names[y_pred[i]]}",
            }
            for i in range(len(texts)) if y_true[i] != y_pred[i]
        ]

        print(f"[ErrorAnalyzer] Total errors: {len(errors)} / {len(texts)} "
              f"({len(errors)/len(texts)*100:.2f}%)")

        aspect_errors = Counter([e['aspect'] for e in errors])
        print(f"\n[ErrorAnalyzer] Error rate by aspect:")
        for aspect in sorted(aspect_errors):
            total = sum(1 for a in aspects if a == aspect)
            print(f"  {aspect:<16}: {aspect_errors[aspect]:>4} / {total} "
                  f"({aspect_errors[aspect]/total*100:.2f}%)")

        error_types = Counter([e['error_type'] for e in errors])
        print(f"\n[ErrorAnalyzer] Error type distribution:")
        for etype, count in sorted(error_types.items(), key=lambda x: -x[1]):
            print(f"  {etype:<25}: {count:>4}  ({count/len(errors)*100:.2f}%)")

        if save_path:
            pd.DataFrame(errors).to_csv(save_path, index=False)
            print(f"\n[ErrorAnalyzer] Detailed errors saved to {save_path}")

        return errors

print("✅ Completed: Defining standalone visualization tools.")


In [3]:
print("⏳ Starting: Defining function evaluate_model...")
from tqdm.auto import tqdm
import time
_start_time = time.time()
def evaluate_model(model, dataloader, config, device, save_dir=None):
    """
    Run the model over all batches in the dataloader and collect predictions.
    Computes per-aspect Macro-F1, overall metrics, MSR analysis, and (optionally) saves outputs.

    Args:
        model:       Trained MultiAspectSentimentModel (or baseline)
        dataloader:  DataLoader for the test or val split
        config:      Full config dict (aspects, model, data settings)
        device:      'cuda' or 'cpu'
        save_dir:    Optional Path to write results (metrics.json, predictions.csv, etc.)

    Returns:
        dict with 'overall', 'per_aspect', and 'mixed_sentiment' metrics
    """
    model.eval()
    all_predictions, all_labels, all_aspects, all_texts, all_review_ids, all_probs = [], [], [], [], [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            aspect_ids     = batch['aspect_ids'].to(device)
            labels         = batch['labels']

            # Pass edge_indices if the model uses GCN (None otherwise to save compute)
            edge_indices = None
            if config['model'].get('use_dependency_gcn', False):
                edge_indices = [e.to(device) if e is not None else None
                                for e in batch['edge_indices']]

            preds = model(input_ids, attention_mask, aspect_ids, edge_indices)
            if isinstance(preds, tuple): preds = preds[0]  # Unpack (logits, attn, repr) tuple

            probs = torch.softmax(preds, dim=1).cpu().numpy()
            all_probs.extend(probs)
            pred_classes = torch.argmax(preds, dim=1).cpu().numpy()
            all_predictions.extend(pred_classes)
            all_labels.extend(labels.numpy())
            all_aspects.extend(batch['aspects'])
            all_texts.extend(batch['texts'])
            all_review_ids.extend(batch['review_ids'])

    # ── Compute metrics ─────────────────────────────────────────────
    evaluator      = AspectSentimentEvaluator(config['aspects']['names'])
    msr_evaluator  = MixedSentimentEvaluator(config['aspects']['names'])
    aspect_metrics = {}

    for aspect in config['aspects']['names']:
        mask  = np.array([a == aspect for a in all_aspects])
        if mask.sum() == 0: continue
        aspect_metrics[aspect] = evaluator.evaluate_aspect(
            np.array(all_labels)[mask],
            np.array(all_predictions)[mask],
            aspect,
            y_prob=np.array(all_probs)[mask]
        )
        print_results(evaluator.results, aspect)
        plot_roc_curve(evaluator.results, aspect, save_path=save_dir / f'roc_{aspect}.png') if save_dir else None

    overall = evaluator.evaluate_aspect(np.array(all_labels), np.array(all_predictions), 'overall', y_prob=np.array(all_probs))

    # ── MSR Analysis ────────────────────────────
    # Group true and predicted labels by review_id for the MSR evaluator
    review_true, review_pred = {}, {}
    for rid, asp, true_lbl, pred_lbl in zip(all_review_ids, all_aspects, all_labels, all_predictions):
        review_true.setdefault(rid, {})[asp] = int(true_lbl)
        review_pred.setdefault(rid, {})[asp] = int(pred_lbl)

    mixed_metrics = msr_evaluator.evaluate_mixed_sentiment_resolution(review_true, review_pred)
    msr_evaluator.print_mixed_sentiment_results(mixed_metrics)

    results = {
        'overall'         : {k: float(v) for k, v in overall.items() if not isinstance(v, (list, np.ndarray))},
        'per_aspect'      : {asp: {k: (v.tolist() if isinstance(v, np.ndarray) else v)
                                    for k, v in mets.items()}
                             for asp, mets in aspect_metrics.items()},
        'mixed_sentiment' : mixed_metrics,
    }

    # ── Save outputs ───────────────────────────
    if save_dir is not None:
        import json, matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        import seaborn as sns

        save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)

        # metrics.json
        print("📝 Writing the final report to disk...")
        (save_dir / 'metrics.json').write_text(
            json.dumps(results, indent=2, default=str), encoding='utf-8'
        )

        # predictions.csv — full table of raw predictions for error analysis
        label_names = {0: 'negative', 1: 'neutral', 2: 'positive'}
        print("💾 Saving progress to a new CSV file...")
        pd.DataFrame({
            'text'      : all_texts,
            'aspect'    : all_aspects,
            'review_id' : all_review_ids,
            'true_label': [label_names.get(l, l) for l in all_labels],
            'pred_label': [label_names.get(p, p) for p in all_predictions],
            'correct'   : [t == p for t, p in zip(all_labels, all_predictions)],
        }).to_csv(save_dir / 'predictions.csv', index=False)

        # error_analysis.csv — only misclassified samples
        print("📖 Reading the dataset from CSV...")
        pred_df = pd.read_csv(save_dir / 'predictions.csv')
        pred_df[pred_df['correct'] == False].to_csv(save_dir / 'error_analysis.csv', index=False)

        # Confusion matrix grid
        aspects = config['aspects']['names']
        n_cols  = 4
        n_rows  = (len(aspects) + n_cols - 1) // n_cols
        print("📈 Preparing some visualizations to show you the results.")
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
        axes_flat = axes.flatten()
        for i, asp in enumerate(aspects):
            if asp in results['per_aspect']:
                cm = np.array(results['per_aspect'][asp]['confusion_matrix'])
                print("📉 Plotting detailed statistical distributions.")
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_flat[i],
                           xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'])
                axes_flat[i].set_title(f'{asp} — Macro-F1: {results["per_aspect"][asp]["macro_f1"]:.3f}',
                                       fontweight='bold')
        for j in tqdm(range(len(aspects), len(axes_flat)), desc="J progress"): axes_flat[j].set_visible(False)
        plt.suptitle('Confusion Matrices per Aspect', fontsize=14, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.savefig(save_dir / 'all_confusion_matrices.png', dpi=150, bbox_inches='tight')
        plot_all_roc_curves(evaluator.results, save_dir=save_dir)
        plt.close()

        # mixed_sentiment_analysis.json
        (save_dir / 'mixed_sentiment_analysis.json').write_text(
            json.dumps(mixed_metrics, indent=2), encoding='utf-8'
        )

        print(f'\nResults saved to: {save_dir}')

    return results
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function evaluate_model.")


⏳ Starting: Defining function evaluate_model...
🕒 Done in 0.00s
✅ Completed: Defining function evaluate_model.


## CLI Entry Point

In [4]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
# ── Run from CLI or modify the paths below to run interactively ──
# python inference_bridge/evaluate.py \
#     --checkpoint outputs/cosmetic_sentiment_v1/best_model.pt \
#     --split test \
#     --save_dir inference_bridge

# ── or interactively: ──
# ckpt    = 'outputs/cosmetic_sentiment_v1/best_model.pt'
# model, config = load_model(ckpt)
# tokenizer = RobertaTokenizer.from_pretrained(config['model']['roberta_model'])
# dataset   = CosmeticReviewDataset(config['data']['test_path'], tokenizer, config, config['aspects']['names'])
# loader    = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_with_dependencies)
# results   = evaluate_model(model, loader, config, device='cuda', save_dir='inference_bridge')

print('evaluate.py loaded. Use CLI or run interactively via the cells above.')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
evaluate.py loaded. Use CLI or run interactively via the cells above.
🕒 Done in 0.00s
✅ Completed: Loading dependencies and libraries.
